In [0]:
import json
import re
import time
import uuid
import socket
import ipaddress
from datetime import date, timedelta
from urllib.parse import urlparse

import requests
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DateType

# ============================================================
# Bronze framework: validates and executes source specs.
# Agents only PROPOSE specs; this deterministic code is the only
# thing that touches the network and the storage.
#   source -> raw files (landing, original format) -> bronze Delta table
# ============================================================

bronze_volume_path = f"/Volumes/{catalog}/{schema}/raw"
ingestion_log_table = f"{catalog}.{schema}.bronze_ingestion_log"

allowed_source_types = {"http_api"}          # future: files, jdbc
allowed_methods = {"GET"}
allowed_formats = {"json", "csv"}
allowed_csv_delimiters = {",", ";", "|", "\t"}
allowed_placeholders = {"start", "end"}
allowed_incremental_types = {"date_range", "snapshot"}
allowed_date_formats = {"%Y-%m-%d", "%Y%m%d", "%d/%m/%Y"}
retry_status_codes = {429, 500, 502, 503, 504}
max_days_back = 730
max_chunk_days = 31
max_response_bytes = 15 * 1024 * 1024        # VARIANT values are limited to 16 MB

# Fixed calendar anchor (a Monday): chunks are always aligned to it, so the same
# dates always fall in the same chunk and the same file -> re-runs overwrite, never duplicate.
# With chunk_days = 7, every chunk is a Monday-Sunday week.
chunk_anchor = date(2000, 1, 3)

bronze_metadata_columns = ["source_name", "endpoint_name", "source_file", "chunk_start", "chunk_end",
                           "run_id", "ingested_at"]

ingestion_log_schema = StructType([
    StructField("run_id", StringType()),
    StructField("source_name", StringType()),
    StructField("endpoint_name", StringType()),
    StructField("chunk_start", DateType()),
    StructField("chunk_end", DateType()),
    StructField("status", StringType()),
    StructField("http_status", IntegerType()),
    StructField("bytes_written", LongType()),
    StructField("file_path", StringType()),
    StructField("error_message", StringType()),
])


# ------------------------------------------------------------
# 1. Validation
# ------------------------------------------------------------
def _int_between(value, low, high):
    return isinstance(value, int) and not isinstance(value, bool) and low <= value <= high


def _placeholders(text):
    return set(re.findall(r"\{([^{}]*)\}", str(text)))


def _is_public_host(hostname):
    try:
        for info in socket.getaddrinfo(hostname, 443):
            ip = ipaddress.ip_address(info[4][0])
            if ip.is_private or ip.is_loopback or ip.is_link_local or ip.is_reserved or ip.is_multicast:
                return False
        return True
    except Exception:
        return False


def validate_source_spec(spec):
    errors = []
    if not isinstance(spec, dict):
        return ["spec must be a mapping (key: value)"]

    # Source
    if not re.fullmatch(r"[a-z][a-z0-9_]{2,40}", str(spec.get("source_name", ""))):
        errors.append("source_name must be snake_case, 3-41 chars")
    if not str(spec.get("description", "")).strip():
        errors.append("description is required")
    if spec.get("source_type") not in allowed_source_types:
        errors.append(f"source_type must be one of {sorted(allowed_source_types)} (other source types are not supported yet)")

    auth = spec.get("auth", {"type": "none"})
    if not isinstance(auth, dict) or auth.get("type") != "none":
        errors.append("only auth type 'none' is supported for now")

    # Incremental strategy
    incremental = spec.get("incremental")
    inc_type = incremental.get("type") if isinstance(incremental, dict) else None
    if inc_type not in allowed_incremental_types:
        errors.append(f"incremental.type must be one of {sorted(allowed_incremental_types)}")
    if inc_type == "date_range":
        if not _int_between(incremental.get("days_back"), 1, max_days_back):
            errors.append(f"incremental.days_back must be an integer between 1 and {max_days_back}")
        if not _int_between(incremental.get("chunk_days"), 1, max_chunk_days):
            errors.append(f"incremental.chunk_days must be an integer between 1 and {max_chunk_days}")
        if incremental.get("date_format", "%Y-%m-%d") not in allowed_date_formats:
            errors.append(f"incremental.date_format must be one of {sorted(allowed_date_formats)}")
        if not _int_between(incremental.get("end_offset_days", 0), -10, 2):
            errors.append("incremental.end_offset_days must be an integer between -10 and 2")

    # Endpoints
    endpoints = spec.get("endpoints")
    if not isinstance(endpoints, list) or not endpoints:
        errors.append("endpoints must be a non-empty list")
        endpoints = []

    seen_names = set()
    for i, ep in enumerate(endpoints):
        if not isinstance(ep, dict):
            errors.append(f"endpoint #{i} must be a mapping")
            continue
        name = str(ep.get("name", ""))
        label = name or f"#{i}"
        if not re.fullmatch(r"[a-z][a-z0-9_]{1,40}", name):
            errors.append(f"endpoint {label}: name must be snake_case")
        if name in seen_names:
            errors.append(f"endpoint {label}: duplicated name")
        seen_names.add(name)

        if str(ep.get("method", "GET")).upper() not in allowed_methods:
            errors.append(f"endpoint {label}: only GET is allowed")
        if ep.get("response_format") not in allowed_formats:
            errors.append(f"endpoint {label}: response_format must be one of {sorted(allowed_formats)}")
        if "csv_delimiter" in ep:
            if ep.get("response_format") != "csv":
                errors.append(f"endpoint {label}: csv_delimiter is only allowed when response_format is csv")
            elif ep["csv_delimiter"] not in allowed_csv_delimiters:
                errors.append(f"endpoint {label}: csv_delimiter must be one of {sorted(allowed_csv_delimiters)}")

        url = str(ep.get("url", ""))
        parsed = urlparse(url)
        if parsed.scheme != "https" or not parsed.hostname:
            errors.append(f"endpoint {label}: url must be a valid https URL")
        elif not _is_public_host(parsed.hostname):
            errors.append(f"endpoint {label}: host {parsed.hostname} is not a public internet host")
        if parsed.query:
            errors.append(f"endpoint {label}: put query parameters in 'params', not in the url")

        params = ep.get("params", {})
        if not isinstance(params, dict):
            errors.append(f"endpoint {label}: params must be a mapping")
            params = {}
        for key, value in params.items():
            if isinstance(value, bool) or not isinstance(value, (str, int, float)):
                errors.append(f"endpoint {label}: param {key} must be a string or a number")

        used = _placeholders(url).union(*[_placeholders(v) for v in params.values()])
        unknown = used - allowed_placeholders
        if unknown:
            errors.append(f"endpoint {label}: unknown placeholders {sorted(unknown)}; only {{start}} and {{end}} are allowed")
        if inc_type == "date_range" and not {"start", "end"} <= used:
            errors.append(f"endpoint {label}: date_range endpoints must use both {{start}} and {{end}}")
        if inc_type == "snapshot" and used:
            errors.append(f"endpoint {label}: snapshot endpoints cannot use date placeholders")
        if "end_offset_days" in ep and not _int_between(ep["end_offset_days"], -10, 2):
            errors.append(f"endpoint {label}: end_offset_days must be an integer between -10 and 2")

    return errors


# ------------------------------------------------------------
# 2. Download helpers (http_api)
# ------------------------------------------------------------
def build_chunks(spec, endpoint, run_date, days_back_override=None):
    """Calendar-aligned chunks: the same dates always produce the same chunk boundaries."""
    incremental = spec["incremental"]
    if incremental["type"] == "snapshot":
        return [(run_date, run_date)]

    days_back = days_back_override or incremental["days_back"]
    chunk_days = incremental["chunk_days"]
    end_offset = endpoint.get("end_offset_days", incremental.get("end_offset_days", 0))

    first_day = run_date - timedelta(days=days_back)
    last_day = run_date + timedelta(days=end_offset)

    # Move first_day back to the start of its aligned chunk
    chunk_start = first_day - timedelta(days=(first_day - chunk_anchor).days % chunk_days)

    chunks = []
    while chunk_start <= last_day:
        aligned_end = chunk_start + timedelta(days=chunk_days - 1)
        chunks.append((chunk_start, min(aligned_end, last_day)))   # current chunk may be partial
        chunk_start = aligned_end + timedelta(days=1)
    return chunks


def _render(value, start_str, end_str):
    if isinstance(value, str):
        return value.replace("{start}", start_str).replace("{end}", end_str)
    return value


def _http_get(url, params, retries=3):
    result = {"status_code": None, "body": None, "error": "not executed"}
    for attempt in range(retries):
        try:
            response = requests.get(
                url, params=params, timeout=60, allow_redirects=False,
                headers={"User-Agent": "databricks-bronze-framework/1.0"},
            )
            result = {"status_code": response.status_code, "body": response.content, "error": None}
            if response.status_code == 200:
                return result
            result["error"] = f"HTTP {response.status_code}: {response.text[:300]}"
            if response.status_code not in retry_status_codes:
                return result
        except requests.RequestException as e:
            result = {"status_code": None, "body": None, "error": str(e)[:300]}
        if attempt < retries - 1:
            time.sleep(2 ** attempt)
    return result


def _check_body(body, response_format):
    if not body:
        return "empty response"
    if len(body) > max_response_bytes:
        return f"response larger than {max_response_bytes} bytes"
    text = body.decode("utf-8", errors="replace")
    if response_format == "json":
        try:
            json.loads(text)
        except ValueError as e:
            return f"invalid JSON: {str(e)[:200]}"
    if response_format == "csv" and len(text.strip().splitlines()) < 2:
        return "CSV has no data rows"
    return None


# ------------------------------------------------------------
# 3. Bronze Delta tables (one per endpoint)
# ------------------------------------------------------------
def bronze_table_name(source_name, endpoint_name):
    return f"{catalog}.{schema}.{source_name}_{endpoint_name}"


def _safe_column_name(name, used):
    base = re.sub(r"[^a-z0-9]+", "_", str(name).strip().lower()).strip("_") or "col"
    if base[0].isdigit():
        base = f"c_{base}"
    candidate, i = base, 2
    while candidate in used:
        candidate = f"{base}_{i}"
        i += 1
    used.add(candidate)
    return candidate


def _file_path_col():
    # _metadata.file_path may come as dbfs:/Volumes/... -> normalize to /Volumes/...
    return F.regexp_replace(F.col("_metadata.file_path"), r"^[a-z]+:", "").alias("source_file")


def load_bronze_tables(spec, results, run_id):
    """Load the files written in this run into bronze Delta tables, replacing previous rows of the same files."""
    loaded = {}
    for ep in spec["endpoints"]:
        files = [r for r in results if r["endpoint_name"] == ep["name"] and r["status"] == "ok"]
        if not files:
            continue

        table = bronze_table_name(spec["source_name"], ep["name"])
        paths = [r["file_path"] for r in files]
        file_info = spark.createDataFrame(
            [(r["file_path"], r["chunk_start"], r["chunk_end"]) for r in files],
            "source_file STRING, chunk_start DATE, chunk_end DATE",
        )

        if ep["response_format"] == "json":
            # One row per file, whole document stored as VARIANT
            df = (spark.read.option("wholetext", True).text(paths)
                  .select(F.expr("parse_json(value)").alias("payload"), _file_path_col()))
        else:
            # One row per CSV line, every column as STRING (typing is silver's job)
            raw = (spark.read.option("header", True).option("inferSchema", False)
                   .option("sep", ep.get("csv_delimiter", ",")).csv(paths))
            used = set(bronze_metadata_columns)
            df = raw.select(
                *[F.col(f"`{c}`").cast("string").alias(_safe_column_name(c, used)) for c in raw.columns],
                _file_path_col(),
            )

        df = (df.join(F.broadcast(file_info), "source_file", "left")
                .withColumn("source_name", F.lit(spec["source_name"]))
                .withColumn("endpoint_name", F.lit(ep["name"]))
                .withColumn("run_id", F.lit(run_id))
                .withColumn("ingested_at", F.current_timestamp()))

        if df.filter(F.col("chunk_start").isNull()).limit(1).count() > 0:
            raise RuntimeError(f"{table}: could not match some rows to their source file")

        data_columns = [c for c in df.columns if c not in bronze_metadata_columns]
        df = df.select(*bronze_metadata_columns, *data_columns)

        if spark.catalog.tableExists(table):
            files_sql = ", ".join(f"'{p}'" for p in paths)
            (df.write.mode("overwrite")
               .option("replaceWhere", f"source_file IN ({files_sql})")   # replace rows of these files only
               .option("mergeSchema", "true")
               .saveAsTable(table))
        else:
            df.write.saveAsTable(table)

        loaded[table] = len(paths)
        print(f"  ↳ bronze table {table}: {len(paths)} file(s) loaded")
    return loaded


# ------------------------------------------------------------
# 4. Run a spec: download -> raw files -> log -> bronze tables
# ------------------------------------------------------------
def run_source_spec(spec, days_back_override=None, dry_run=False, run_date=None):
    errors = validate_source_spec(spec)
    if errors:
        raise ValueError(f"Invalid source spec: {errors}")

    run_date = run_date or date.today()
    run_id = str(uuid.uuid4())
    source_name = spec["source_name"]
    date_format = spec["incremental"].get("date_format", "%Y-%m-%d")
    results = []

    if not dry_run:
        spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.raw")

    for ep in spec["endpoints"]:
        fmt = ep["response_format"]
        folder = f"{bronze_volume_path}/{source_name}/{ep['name']}"
        if not dry_run:
            dbutils.fs.mkdirs(folder)

        for chunk_start, chunk_end in build_chunks(spec, ep, run_date, days_back_override):
            start_str, end_str = chunk_start.strftime(date_format), chunk_end.strftime(date_format)
            url = _render(ep["url"], start_str, end_str)
            params = {k: _render(v, start_str, end_str) for k, v in ep.get("params", {}).items()}

            response = _http_get(url, params)
            error = response["error"] or _check_body(response["body"], fmt)
            file_path, bytes_written, sample = None, 0, None

            if not error:
                text = response["body"].decode("utf-8", errors="replace")
                sample = text[:1500]
                if not dry_run:
                    # File name = aligned chunk start -> same chunk, same file, always overwritten
                    file_path = f"{folder}/{ep['name']}_{chunk_start}.{fmt}"
                    dbutils.fs.put(file_path, text, overwrite=True)
                    bytes_written = len(response["body"])

            status = "error" if error else ("dry_run_ok" if dry_run else "ok")
            print(f"{source_name}.{ep['name']} {chunk_start}->{chunk_end}: {status}"
                  + (f" ({error[:120]})" if error else ""))
            results.append({
                "endpoint_name": ep["name"], "chunk_start": chunk_start, "chunk_end": chunk_end,
                "status": status, "http_status": response["status_code"], "bytes_written": bytes_written,
                "file_path": file_path, "error_message": error, "sample": sample,
            })

    if not dry_run and results:
        log_rows = [(run_id, source_name, r["endpoint_name"], r["chunk_start"], r["chunk_end"], r["status"],
                     r["http_status"], r["bytes_written"], r["file_path"], r["error_message"]) for r in results]
        (spark.createDataFrame(log_rows, ingestion_log_schema)
              .withColumn("ingested_at", F.current_timestamp())
              .write.mode("append").saveAsTable(ingestion_log_table))

    bronze_tables = {}
    if not dry_run and any(r["status"] == "ok" for r in results):
        try:
            bronze_tables = load_bronze_tables(spec, results, run_id)
        except Exception as e:
            raise RuntimeError(
                f"Raw files were downloaded and logged (run_id {run_id}), but loading bronze tables failed: {e}"
            )

    return {
        "run_id": run_id,
        "source_name": source_name,
        "chunks_ok": sum(1 for r in results if r["status"] != "error"),
        "chunks_error": sum(1 for r in results if r["status"] == "error"),
        "bronze_tables": bronze_tables,
        "results": results,
    }

In [0]:
import yaml

# ------------------------------------------------------------
# 4. YAML support: humans read/write YAML, the framework works with dicts
# ------------------------------------------------------------
def load_source_spec(yaml_text):
    """Parse a YAML source spec into a dict."""
    try:
        spec = yaml.safe_load(yaml_text)
    except yaml.YAMLError as e:
        raise ValueError(f"Invalid YAML: {str(e)[:300]}")
    if not isinstance(spec, dict):
        raise ValueError("YAML spec must be a mapping (key: value) at the top level")
    return spec


def spec_to_yaml(spec):
    """Render a spec dict as readable YAML (for review and approval)."""
    return yaml.safe_dump(spec, sort_keys=False, allow_unicode=True, default_flow_style=False)

In [0]:
import os

# ------------------------------------------------------------
# 5. Spec files: one YAML file per source in the source_specs/ folder (versioned in Git)
# ------------------------------------------------------------
def find_source_specs_dir(folder_name="source_specs"):
    """Walk up from the running notebook's folder until source_specs/ is found."""
    current = os.getcwd()
    while True:
        candidate = os.path.join(current, folder_name)
        if os.path.isdir(candidate):
            return candidate
        parent = os.path.dirname(current)
        if parent == current:
            raise FileNotFoundError(f"Folder '{folder_name}' not found from {os.getcwd()} upwards")
        current = parent


def list_source_specs():
    return sorted(f[:-4] for f in os.listdir(find_source_specs_dir()) if f.endswith(".yml"))


def load_source_spec_file(source_name):
    path = os.path.join(find_source_specs_dir(), f"{source_name}.yml")
    with open(path, encoding="utf-8") as f:
        spec = load_source_spec(f.read())
    if spec.get("source_name") != source_name:
        raise ValueError(f"{path}: source_name '{spec.get('source_name')}' must match the file name")
    return spec


def save_source_spec_file(spec):
    """Validate and write a spec to source_specs/<source_name>.yml (used by the agent after approval)."""
    errors = validate_source_spec(spec)
    if errors:
        raise ValueError(f"Invalid source spec, not saved: {errors}")
    path = os.path.join(find_source_specs_dir(), f"{spec['source_name']}.yml")
    with open(path, "w", encoding="utf-8") as f:
        f.write(spec_to_yaml(spec))
    return path

In [0]:
# ------------------------------------------------------------
# 6. probe_url: safe exploratory GET for agents (never stores anything)
# ------------------------------------------------------------
max_probe_bytes = 2 * 1024 * 1024
max_probes_per_session = 20
probe_calls = {"count": 0}


def reset_probe_budget():
    probe_calls["count"] = 0


def _first_informative(items):
    """First list element that is not null/empty, to describe the list."""
    for item in items:
        if item not in (None, "", [], {}):
            return item
    return items[0]


def _json_outline(value, depth=0, max_depth=10, max_keys=25):
    """Compact structure of a JSON document, with example values, for an LLM to read."""
    if depth >= max_depth:
        return "..."
    if isinstance(value, dict):
        outline = {k: _json_outline(v, depth + 1) for k, v in list(value.items())[:max_keys]}
        if len(value) > max_keys:
            outline["..."] = f"{len(value) - max_keys} more keys"
        return outline
    if isinstance(value, list):
        if not value:
            return "array[0]"
        return {f"array[{len(value)}] of": _json_outline(_first_informative(value), depth + 1)}
    if isinstance(value, bool):
        return f"bool (e.g. {value})"
    if isinstance(value, (int, float)):
        return f"number (e.g. {value})"
    if value is None:
        return "null"
    return f"string (e.g. {str(value)[:40]!r})"


def _detect_format(text, content_type):
    stripped = text.lstrip()
    content_type = content_type.lower()
    if "json" in content_type or stripped[:1] in ("{", "["):
        try:
            json.loads(text)
            return "json"
        except ValueError:
            pass
    if "html" in content_type or "xml" in content_type or stripped[:1] == "<":
        return "html_or_xml"
    lines = text.splitlines()
    if len(lines) >= 2 and any(d in lines[0] for d in allowed_csv_delimiters):
        return "csv"
    return "text"


def probe_url(url, params=None):
    """Exploratory GET with the same safety rules as the framework. Returns a compact description."""
    params = params or {}
    errors = []
    parsed = urlparse(str(url))
    if parsed.scheme != "https" or not parsed.hostname:
        errors.append("url must be a valid https URL")
    elif not _is_public_host(parsed.hostname):
        errors.append(f"host {parsed.hostname} is not a public internet host")
    if parsed.query:
        errors.append("put query parameters in 'params', not in the url")
    if not isinstance(params, dict):
        errors.append("params must be an object")
        params = {}
    for key, value in params.items():
        if isinstance(value, bool) or not isinstance(value, (str, int, float)):
            errors.append(f"param {key} must be a string or a number")
    if errors:
        return {"status": "rejected", "errors": errors}

    if probe_calls["count"] >= max_probes_per_session:
        return {"status": "rejected",
                "errors": [f"probe budget exhausted ({max_probes_per_session} calls); "
                           "propose a spec with what you know, or ask the user"]}
    probe_calls["count"] += 1

    try:
        response = requests.get(
            url, params=params, timeout=30, allow_redirects=False, stream=True,
            headers={"User-Agent": "databricks-bronze-framework/1.0"},
        )
        body = response.raw.read(max_probe_bytes + 1, decode_content=True)
        response.close()
    except requests.RequestException as e:
        return {"status": "error", "error": str(e)[:300]}

    result = {
        "status": "ok" if response.status_code == 200 else "http_error",
        "http_status": response.status_code,
        "requested_url": response.url,
        "content_type": response.headers.get("Content-Type", ""),
        "bytes_read": min(len(body), max_probe_bytes),
        "truncated": len(body) > max_probe_bytes,
        "probes_left": max_probes_per_session - probe_calls["count"],
    }

    if 300 <= response.status_code < 400:
        result["redirect_to"] = response.headers.get("Location")
        result["hint"] = "Redirects are not followed. Probe redirect_to directly if you trust it (it will be validated)."
        return result

    text = body[:max_probe_bytes].decode("utf-8", errors="replace")
    if result["truncated"]:
        result["hint"] = "Response is larger than the probe limit: narrow the date range or filters."
    else:
        detected = _detect_format(text, result["content_type"])
        result["detected_format"] = detected
        if detected == "json":
            result["structure"] = _json_outline(json.loads(text))
        elif detected == "csv":
            lines = text.splitlines()
            result["csv_delimiter_guess"] = max(allowed_csv_delimiters, key=lambda d: lines[0].count(d))
            result["csv_preview"] = lines[:6]
    result["sample"] = text[:1500]
    return result